# 🚀 50M Bengali GPT — ২-স্টেজ প্রোডাকশন ট্রেনিং
### 📊 স্টেপ ক্যালকুলেশন:
```
Tokens/Step  = batch(4) × grad_accum(4) × context(512) = 8,192
Corpus Lines ≈ 11 লাখ (Wikipedia + News + Alpaca + Math + 10 Niche)
Train Tokens ≈ 11,00,000 × 30 × 0.9 ≈ 2.97 কোটি
Steps/Epoch  ≈ 2.97 কোটি ÷ 8,192 ≈ 3,625 steps
Total (2 ep) ≈ 3,625 × 2 = ~7,250 steps  (~35 min on T4)
```
**নিশ ডোমেন (১,০০,০০০ লাইন):**
ডিজিটাল মার্কেটিং • NCTB শিক্ষা • স্বাস্থ্য • ব্যবসা • প্রযুক্তি • কৃষি • ইসলামিক • রান্না • আইন • সাধারণ জ্ঞান


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BASE_DIR = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints'
STAGE1_DIR = os.path.join(DRIVE_BASE_DIR, 'stage1_pretrain')
STAGE2_DIR = os.path.join(DRIVE_BASE_DIR, 'stage2_sft')
os.makedirs(STAGE1_DIR, exist_ok=True)
os.makedirs(STAGE2_DIR, exist_ok=True)
print(f'✓ Drive রেডি: {DRIVE_BASE_DIR}')

In [ ]:
%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git
%cd /content/ss_100m/ss_50million
!pip install -q -r requirements.txt
print('✓ পরিবেশ প্রস্তুত!')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Step 4: বিশাল ডেটাসেট তৈরি (~১১ লাখ লাইন)
# Sources:
#   ১. বাংলা Wikipedia   → ৫,০০,০০০
#   ২. বাংলা News         → ২,০০,০০০
#   ৩. English Wikipedia  → ১,০০,০০০
#   ৪. Alpaca-Orca         →    ৮০,০০০
#   ৫. নিশ ডোমেন (১০টি)   → ১,০০,০০০
#   ৬. পাটিগণিত            → ১,০০,০০০
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, re, json, random
from datasets import load_dataset

os.makedirs('data', exist_ok=True)
corpus_path   = 'data/corpus.txt'
sft_data_path = 'data/sft_data.txt'

need_download = True
if os.path.exists(corpus_path) and os.path.exists(sft_data_path):
    with open(corpus_path, encoding='utf-8') as f: c_cnt = sum(1 for _ in f)
    with open(sft_data_path, encoding='utf-8') as f: s_cnt = sum(1 for _ in f)
    if c_cnt >= 1000000 and s_cnt >= 150000:
        print(f'✓ ডেটা বিদ্যমান (Corpus:{c_cnt:,} | SFT:{s_cnt:,}) — স্কিপ।')
        need_download = False
    else:
        print(f'⚠️ ছোট (Corpus:{c_cnt:,}, SFT:{s_cnt:,}) — রিডাউনলোড হচ্ছে...')

if need_download:
    corpus_lines, sft_lines = [], []
    print('='*70)
    print('📥 ১১ লাখ লাইনের ডেটা সংগ্রহ শুরু হচ্ছে...')
    print('='*70)

    # ━━━ SOURCE ১: বাংলা Wikipedia (৫,০০,০০০) ━━━━━━━━━━━━━━━━━━━━━━━
    print('  [১/৬] বাংলা Wikipedia → ৫,০০,০০০...')
    wiki_bn = load_dataset('wikimedia/wikipedia','20231101.bn',split='train',streaming=True)
    bn_cnt  = 0
    for item in wiki_bn:
        for p in item.get('text','').split('\n'):
            p = p.strip()
            if len(p)>=25 and re.search(r'[\u0980-\u09FF]',p):
                corpus_lines.append(p); bn_cnt+=1
                if bn_cnt>=500000: break
        if bn_cnt>=500000: break
    print(f'     ✓ {bn_cnt:,} লাইন')

    # ━━━ SOURCE ২: বাংলা News (২,০০,০০০) ━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print('  [২/৬] বাংলা Newspaper → ২,০০,০০০...')
    try:
        news_ds  = load_dataset('zabir-nabil/bangla_newspaper_dataset',split='train',streaming=True)
        news_cnt = 0
        for row in news_ds:
            txt = (row.get('text','') or row.get('content','')).strip()
            for p in txt.split('\n'):
                p = p.strip()
                if len(p)>=25 and re.search(r'[\u0980-\u09FF]',p):
                    corpus_lines.append(p); news_cnt+=1
                    if news_cnt>=200000: break
            if news_cnt>=200000: break
        print(f'     ✓ {news_cnt:,} লাইন')
    except Exception as e:
        print(f'     ⚠️ News ফলব্যাক: {e}')

    # ━━━ SOURCE ৩: English Wikipedia (১,০০,০০০) ━━━━━━━━━━━━━━━━━━━━━
    print('  [৩/৬] English Wikipedia → ১,০০,০০০...')
    wiki_en = load_dataset('wikimedia/wikipedia','20231101.en',split='train',streaming=True)
    en_cnt  = 0
    for item in wiki_en:
        for p in item.get('text','').split('\n'):
            p = p.strip()
            if len(p)>=35 and re.search(r'[a-zA-Z]',p):
                corpus_lines.append(p); en_cnt+=1
                if en_cnt>=100000: break
        if en_cnt>=100000: break
    print(f'     ✓ {en_cnt:,} লাইন')

    # ━━━ SOURCE ৪: Alpaca-Orca (৮০,০০০) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print('  [৪/৬] Bangla Alpaca-Orca → ৮০,০০০...')
    try:
        alpaca_ds = load_dataset('BanglaLLM/bangla-alpaca-orca',split='train',streaming=True)
        alp_cnt   = 0
        for row in alpaca_ds:
            inst = row.get('instruction','').strip()
            inp  = row.get('input','').strip()
            out  = row.get('output','').strip()
            if inst and out:
                fq = f'{inst} {inp}'.strip()
                sft_lines.append(f'প্রশ্ন: {fq} উত্তর: {out} <EOS>')
                corpus_lines.append(f'{fq} {out}')
                alp_cnt+=1
                if alp_cnt>=80000: break
        print(f'     ✓ {alp_cnt:,} জোড়া')
    except Exception as e:
        print(f'     ⚠️ Alpaca ফলব্যাক: {e}')

    # ━━━ SOURCE ৫: নিশ ডোমেন — ১০ ক্যাটাগরি × ১০,০০০ = ১,০০,০০০ ━━━
    print('  [৫/৬] নিশ ডোমেন (১০ ক্যাটাগরি × ১০,০০০) → ১,০০,০০০...')

    # লোকাল JSONL (থাকলে)
    for jf in ['data/digital_marketing.jsonl',
               '/content/drive/MyDrive/digital_marketing.jsonl',
               'data/nctb_data.jsonl']:
        if os.path.exists(jf):
            with open(jf,encoding='utf-8') as f:
                for line in f:
                    try:
                        d = json.loads(line.strip())
                        i2,o2 = d.get('instruction','').strip(), d.get('output','').strip()
                        if i2 and o2:
                            corpus_lines.append(f'{i2} {o2}')
                            sft_lines.append(f'প্রশ্ন: {i2} উত্তর: {o2} <EOS>')
                    except: pass
            print(f'     ✓ JSONL লোড: {jf}')

    def _add_niche(qa_pairs, n_each):
        """প্রতিটি QA জোড়া n_each বার করে corpus ও sft-এ যোগ করে।"""
        for q, a in qa_pairs:
            for _ in range(n_each):
                corpus_lines.append(f'{q} {a}')
                sft_lines.append(f'প্রশ্ন: {q} উত্তর: {a} <EOS>')

    # ── ১. ডিজিটাল মার্কেটিং (১০,০০০) ──────────────────────────────
    dm_qa = [
        ('ডিজিটাল মার্কেটিং কী?','ডিজিটাল মার্কেটিং হলো ইন্টারনেট ও ডিজিটাল মাধ্যম ব্যবহার করে পণ্য বা সেবার প্রচার ও বিক্রি করার প্রক্রিয়া।'),
        ('SEO কী?','SEO (Search Engine Optimization) হলো ওয়েবসাইটকে গুগলের মতো সার্চ ইঞ্জিনে উপরে দেখানোর কৌশল।'),
        ('Facebook Ads কিভাবে কাজ করে?','Facebook Ads টার্গেট অডিয়েন্স নির্বাচন করে, বাজেট নির্ধারণ করে এবং ফেসবুক/ইনস্টাগ্রামে বিজ্ঞাপন দেখায়।'),
        ('CTR কী?','CTR (Click-Through Rate) হলো মোট ইম্প্রেশনের মধ্যে কতজন বিজ্ঞাপনে ক্লিক করেছে তার শতকরা হার।'),
        ('কনটেন্ট মার্কেটিং কী?','কনটেন্ট মার্কেটিং হলো মূল্যবান ও প্রাসঙ্গিক কনটেন্ট তৈরি করে টার্গেট অডিয়েন্স আকৃষ্ট করার কৌশল।'),
        ('ইমেইল মার্কেটিং কেন গুরুত্বপূর্ণ?','ইমেইল মার্কেটিং সরাসরি গ্রাহকের কাছে পৌঁছায়, ROI সবচেয়ে বেশি এবং ব্যক্তিগতকৃত বার্তা পাঠানো যায়।'),
        ('অ্যাফিলিয়েট মার্কেটিং কী?','অ্যাফিলিয়েট মার্কেটিং হলো অন্যের পণ্য প্রচার করে বিক্রির কমিশন উপার্জন করার পদ্ধতি।'),
        ('সোশ্যাল মিডিয়া মার্কেটিং কোন প্ল্যাটফর্মে সেরা?','বাংলাদেশে ফেসবুক সবচেয়ে কার্যকর, তারপর ইউটিউব ও ইনস্টাগ্রাম জনপ্রিয়।'),
        ('গুগল অ্যানালিটিক্স কী কাজে লাগে?','গুগল অ্যানালিটিক্স ওয়েবসাইটে ভিজিটর কোথা থেকে আসছে, কোন পেজ দেখছে এবং কতক্ষণ থাকছে তা ট্র্যাক করে।'),
        ('ROAS কী?','ROAS (Return on Ad Spend) হলো বিজ্ঞাপনে ১ টাকা খরচ করলে কত টাকা আয় হয় তার পরিমাপ।'),
        ('ল্যান্ডিং পেজ কী?','ল্যান্ডিং পেজ হলো বিজ্ঞাপনে ক্লিক করলে যে বিশেষ পেজে ভিজিটর আসে, যা তাকে কেনা বা সাইনআপ করতে উৎসাহিত করে।'),
        ('ইনফ্লুয়েন্সার মার্কেটিং কী?','ইনফ্লুয়েন্সার মার্কেটিং হলো সোশ্যাল মিডিয়ায় জনপ্রিয় ব্যক্তিদের দিয়ে পণ্যের প্রচার করা।'),
        ('কিওয়ার্ড রিসার্চ কেন দরকার?','কিওয়ার্ড রিসার্চ জানায় মানুষ কোন শব্দ দিয়ে সার্চ করে, যা দিয়ে SEO কনটেন্ট তৈরি করা যায়।'),
        ('ব্যাকলিংক কী?','ব্যাকলিংক হলো অন্য ওয়েবসাইট থেকে আপনার সাইটে আসা লিংক যা গুগলের কাছে বিশ্বাসযোগ্যতা বাড়ায়।'),
        ('কনভার্সন রেট কী?','কনভার্সন রেট হলো মোট ভিজিটরের মধ্যে কতজন কাঙ্ক্ষিত কাজ (কেনা/সাইনআপ) করেছে তার শতকরা হার।'),
        ('পেইড মার্কেটিং ও অর্গানিক মার্কেটিংয়ের পার্থক্য কী?','পেইড মার্কেটিং টাকা দিয়ে দ্রুত ট্র্যাফিক আনে। অর্গানিক মার্কেটিং বিনামূল্যে কিন্তু সময় লাগে।'),
        ('ফেসবুক পেজ বুস্ট কিভাবে করব?','ফেসবুক পেজে "Boost Post" ক্লিক করুন, অডিয়েন্স ও বাজেট নির্ধারণ করুন এবং পেমেন্ট দিন।'),
        ('ই-কমার্সে পণ্যের ছবি কেমন হওয়া উচিত?','সাদা ব্যাকগ্রাউন্ড, উচ্চ রেজোলিউশন, একাধিক অ্যাঙ্গেল, পণ্যের বৈশিষ্ট্য স্পষ্ট দেখা যায় এমন ছবি।'),
        ('YouTube চ্যানেল মনিটাইজ কিভাবে করব?','১,০০০ সাবস্ক্রাইবার ও ৪,০০০ ঘণ্টা ওয়াচটাইম পূরণ করুন, তারপর YouTube Partner Program-এ আবেদন করুন।'),
        ('গ্রাহকের রিভিউ কিভাবে পাব?','পণ্য ডেলিভারির পর ফলোআপ মেসেজ পাঠান, রিভিউ দেওয়া সহজ করুন, ভালো রিভিউয়ে ছোট পুরস্কার দিন।'),
    ]
    _add_niche(dm_qa, 500)
    print(f'     ✓ ডিজিটাল মার্কেটিং: {len(dm_qa)*500:,} লাইন')

    # ── ২. NCTB শিক্ষা (১০,০০০) ──────────────────────────────────────
    nctb_qa = [
        ('সালোকসংশ্লেষণ কী?','সালোকসংশ্লেষণ হলো উদ্ভিদের সূর্যালোক, পানি ও CO₂ ব্যবহার করে গ্লুকোজ ও অক্সিজেন তৈরির প্রক্রিয়া।'),
        ('মুক্তিযুদ্ধ কত সালে হয়েছিল?','১৯৭১ সালে। ২৬ মার্চ স্বাধীনতা ঘোষণা ও ১৬ ডিসেম্বর বিজয় দিবস।'),
        ('পিথাগোরাসের উপপাদ্য কী?','সমকোণী ত্রিভুজে অতিভুজের বর্গ = অন্য দুই বাহুর বর্গের সমষ্টি। a² + b² = c²।'),
        ('বাংলাদেশের জাতীয় ফুল কী?','শাপলা (সাদা রঙের) বাংলাদেশের জাতীয় ফুল।'),
        ('অ্যাসিড ও ক্ষার কী?','অ্যাসিড H⁺ দানকারী, pH ৭-এর নিচে। ক্ষার OH⁻ দানকারী, pH ৭-এর উপরে।'),
        ('বঙ্গবন্ধু কে ছিলেন?','বঙ্গবন্ধু শেখ মুজিবুর রহমান বাংলাদেশের স্থপতি ও প্রথম রাষ্ট্রপতি।'),
        ('ক্লোরোফিল কী?','ক্লোরোফিল উদ্ভিদের সবুজ রঙ্গক যা সূর্যালোক শোষণ করে সালোকসংশ্লেষণে সাহায্য করে।'),
        ('বাংলাদেশের মোট জেলা কয়টি?','বাংলাদেশে ৬৪টি জেলা ও ৮টি বিভাগ।'),
        ('আলোর প্রতিফলন কী?','আলো মসৃণ তলে পড়লে ফিরে আসে। আপতন কোণ = প্রতিফলন কোণ।'),
        ('কারক কত প্রকার?','বাংলা ব্যাকরণে কারক ৬ প্রকার: কর্তৃ, কর্ম, করণ, সম্প্রদান, অপাদান ও অধিকরণ।'),
        ('DNA কী?','DNA (Deoxyribonucleic Acid) জীবের বংশগতির তথ্য বহনকারী দ্বিসূত্রী হেলিক্স অণু।'),
        ('বায়ুমণ্ডলের স্তর কয়টি?','৫টি: ট্রপোস্ফিয়ার, স্ট্র্যাটোস্ফিয়ার, মেসোস্ফিয়ার, থার্মোস্ফিয়ার, এক্সোস্ফিয়ার।'),
        ('বাংলা স্বরবর্ণ কয়টি?','বাংলা স্বরবর্ণ ১১টি: অ আ ই ঈ উ ঊ ঋ এ ঐ ও ঔ।'),
        ('নিউটনের প্রথম সূত্র কী?','বাহ্যিক বল না থাকলে স্থির বস্তু স্থির ও গতিশীল বস্তু সমবেগে চলতে থাকে।'),
        ('বাংলাদেশের সংবিধান কবে কার্যকর হয়?','১৯৭২ সালের ১৬ ডিসেম্বর থেকে বাংলাদেশের সংবিধান কার্যকর হয়।'),
        ('ত্রিভুজের ক্ষেত্রফল কত?','ত্রিভুজের ক্ষেত্রফল = ½ × ভূমি × উচ্চতা।'),
        ('আলোর বেগ কত?','আলোর বেগ শূন্যে প্রায় ৩×১০⁸ মিটার/সেকেন্ড।'),
        ('অক্সিজেনের রাসায়নিক প্রতীক কী?','অক্সিজেনের রাসায়নিক প্রতীক O এবং পারমাণবিক সংখ্যা ৮।'),
        ('মুঘল সাম্রাজ্য কে প্রতিষ্ঠা করেন?','বাবর ১৫২৬ সালে পানিপথের প্রথম যুদ্ধে জয়ী হয়ে মুঘল সাম্রাজ্য প্রতিষ্ঠা করেন।'),
        ('বাংলাদেশের জাতীয় পশু কী?','বাংলাদেশের জাতীয় পশু রয়েল বেঙ্গল টাইগার।'),
    ]
    _add_niche(nctb_qa, 500)
    print(f'     ✓ NCTB শিক্ষা: {len(nctb_qa)*500:,} লাইন')

    # ── ৩. স্বাস্থ্য ও চিকিৎসা (১০,০০০) ────────────────────────────
    health_qa = [
        ('ডায়াবেটিস কী?','ডায়াবেটিস হলো রক্তে শর্করার মাত্রা বেশি থাকার রোগ। টাইপ ১ ও টাইপ ২ দুই প্রকার।'),
        ('উচ্চ রক্তচাপ নিয়ন্ত্রণে কী করব?','লবণ কম খান, নিয়মিত ব্যায়াম করুন, স্ট্রেস কমান ও ডাক্তারের পরামর্শে ওষুধ খান।'),
        ('ভিটামিন ডি কোথায় পাওয়া যায়?','সূর্যের আলো প্রধান উৎস। মাছের তেল, ডিমের কুসুম ও দুধ থেকেও পাওয়া যায়।'),
        ('ডেঙ্গু জ্বরের লক্ষণ কী?','তীব্র জ্বর, মাথাব্যথা, চোখের পেছনে ব্যথা, গায়ে ব্যথা ও র‍্যাশ ডেঙ্গুর প্রধান লক্ষণ।'),
        ('প্রতিদিন কতটুকু পানি পান করা উচিত?','প্রাপ্তবয়স্কদের প্রতিদিন ৮-১০ গ্লাস (২-২.৫ লিটার) পানি পান করা উচিত।'),
        ('কোলেস্টেরল কমানোর উপায়?','তৈলাক্ত খাবার কমান, শাকসবজি-ফল বাড়ান, নিয়মিত হাঁটুন, ওমেগা-৩ মাছ খান।'),
        ('শিশুকে কখন টিকা দিতে হয়?','জন্মে BCG-পোলিও, ৬ সপ্তাহে DPT, ৯ মাসে হাম, ১৫ মাসে MMR।'),
        ('ক্যান্সারের প্রাথমিক লক্ষণ কী?','অস্বাভাবিক মাংসপিণ্ড, দীর্ঘস্থায়ী কাশি, অকারণ ওজন কমা ও ক্ষত না শুকানো।'),
        ('মানসিক স্বাস্থ্য ভালো রাখার উপায়?','নিয়মিত ঘুম, ব্যায়াম, পরিবারের সাথে সময় ও প্রয়োজনে বিশেষজ্ঞের পরামর্শ।'),
        ('থাইরয়েড সমস্যার লক্ষণ কী?','হাইপো: ওজন বৃদ্ধি, ক্লান্তি। হাইপার: ওজন কমা, ঘাম ও হৃদস্পন্দন বৃদ্ধি।'),
        ('রক্তস্বল্পতা কেন হয়?','আয়রনের অভাব, ভিটামিন B12 ঘাটতি বা দীর্ঘস্থায়ী রোগে রক্তস্বল্পতা হয়। শাকসবজি ও মাংস খান।'),
        ('গ্যাস্ট্রিক সমস্যার সমাধান কী?','নিয়মিত খাবার খান, মশলাদার-ভাজা খাবার কমান, পানি বেশি পান, ধূমপান পরিহার করুন।'),
        ('হার্ট অ্যাটাকের লক্ষণ কী?','বুকে তীব্র ব্যথা, বাম হাতে ব্যথা, শ্বাসকষ্ট ও ঘাম হওয়া। তুরন্ত হাসপাতালে যান।'),
        ('ঘুমের সমস্যা হলে কী করব?','নির্দিষ্ট সময়ে ঘুমান, ঘুমের আগে স্ক্রিন এড়ান, ক্যাফেইন কমান ও হালকা ব্যায়াম করুন।'),
        ('ওজন কমানোর সহজ উপায় কী?','ক্যালরি ঘাটতি তৈরি করুন, প্রোটিন বাড়ান, চিনি কমান, নিয়মিত ৩০ মিনিট হাঁটুন।'),
        ('সর্দি-কাশিতে কী করব?','বিশ্রাম নিন, গরম পানি-লেবু-মধু পান করুন, বাষ্প নিন। ৩ দিনের বেশি থাকলে ডাক্তার দেখান।'),
        ('চোখের দৃষ্টি ভালো রাখার উপায়?','সবুজ শাকসবজি, গাজর ও মাছ খান, স্ক্রিন টাইম কমান, ৬ মাস অন্তর চোখ পরীক্ষা করুন।'),
        ('কিডনি ভালো রাখতে কী করব?','পর্যাপ্ত পানি পান করুন, লবণ-প্রোটিন কমান, ব্যথানাশক বেশি না খান, ডায়াবেটিস নিয়ন্ত্রণ রাখুন।'),
        ('হাড় শক্ত রাখার উপায় কী?','ক্যালসিয়াম (দুধ, দই) ও ভিটামিন ডি নিন, সূর্যের আলোতে থাকুন, ওজন বহনকারী ব্যায়াম করুন।'),
        ('এলার্জির চিকিৎসা কী?','এলার্জির উৎস এড়িয়ে চলুন, অ্যান্টিহিস্টামিন ওষুধ নিন, গুরুতর হলে ডাক্তারের পরামর্শ নিন।'),
    ]
    _add_niche(health_qa, 500)
    print(f'     ✓ স্বাস্থ্য: {len(health_qa)*500:,} লাইন')

    # ── ৪. ব্যবসা ও উদ্যোক্তা (১০,০০০) ─────────────────────────────
    biz_qa = [
        ('ব্যবসায়িক পরিকল্পনা কিভাবে লিখব?','সারসংক্ষেপ, বাজার বিশ্লেষণ, পণ্য বিবরণ, মার্কেটিং কৌশল ও আর্থিক পরিকল্পনা অন্তর্ভুক্ত করুন।'),
        ('উদ্যোক্তা হওয়ার প্রথম ধাপ কী?','সমস্যা খুঁজুন, সমাধান ভাবুন, ছোট পরিসরে পরীক্ষা করুন, মার্কেট ভ্যালিডেট করুন।'),
        ('ট্রেড লাইসেন্স কিভাবে নেব?','স্থানীয় সিটি কর্পোরেশনে আবেদন করুন। NID, ছবি, ভাড়ার চুক্তি ও ফি লাগে।'),
        ('ক্যাশ ফ্লো কী?','ব্যবসায় প্রতি মাসে কত টাকা ঢুকছে ও কত বের হচ্ছে তার হিসাব।'),
        ('e-Commerce ব্যবসা কিভাবে শুরু করব?','পণ্য নির্বাচন, সাপ্লায়ার, ফেসবুক পেজ, পেমেন্ট গেটওয়ে ও ডেলিভারি পার্টনার ঠিক করুন।'),
        ('ব্র্যান্ডিং কী?','ব্যবসার পরিচয়, লোগো, রং ও মূল্যবোধ দিয়ে গ্রাহকের মনে একটি ছাপ তৈরি করা।'),
        ('মূলধন কিভাবে সংগ্রহ করব?','নিজের সঞ্চয়, পরিবার, ব্যাংক ঋণ, মাইক্রোফাইন্যান্স বা এঞ্জেল ইনভেস্টর থেকে।'),
        ('লাভজনক নিশ কোনগুলো?','হোম ডেলিভারি ফুড, অনলাইন ফ্যাশন, ডিজিটাল সার্ভিস, কৃষি পণ্য ও গৃহস্থালি সেবা।'),
        ('গ্রাহক ধরে রাখার কৌশল?','মানসম্পন্ন পণ্য, দ্রুত সেবা, আফটার-সেলস সাপোর্ট ও নিয়মিত যোগাযোগ।'),
        ('নেগোশিয়েশন কৌশল কী?','প্রস্তুতি নিন, উভয়ের লাভ ভাবুন, ধৈর্য রাখুন ও সর্বনিম্ন সীমা জানুন।'),
        ('ব্যবসার লস কমানোর উপায়?','খরচ ট্র্যাক করুন, অপ্রয়োজনীয় ব্যয় কমান, বিক্রয় বাড়ান এবং দেনা দ্রুত আদায় করুন।'),
        ('ফ্র্যাঞ্চাইজ ব্যবসা কী?','অন্যের প্রতিষ্ঠিত ব্র্যান্ড ও মডেল ব্যবহার করে ব্যবসা করার ব্যবস্থা। যেমন KFC, Pizza Hut।'),
        ('B2B ও B2C পার্থক্য কী?','B2B হলো ব্যবসা থেকে ব্যবসায় বিক্রি। B2C হলো সরাসরি ভোক্তার কাছে বিক্রি।'),
        ('পণ্যের মূল্য নির্ধারণ কিভাবে করব?','উৎপাদন খরচ + মুনাফা মার্জিন। প্রতিযোগীর মূল্য দেখুন এবং বাজারের চাহিদা বুঝুন।'),
        ('গ্রাহক অভিযোগ সামলানোর উপায়?','শুনুন, মাফ চান, দ্রুত সমাধান দিন এবং ভবিষ্যতে যেন না হয় সে ব্যবস্থা নিন।'),
        ('সাপ্লাই চেইন কী?','পণ্য উৎপাদন থেকে শুরু করে গ্রাহকের কাছে পৌঁছানো পর্যন্ত সকল ধাপের শৃঙ্খল।'),
        ('ইনভেন্টরি ব্যবস্থাপনা কেন দরকার?','পণ্যের মজুত পরিমাণ জানা ও সঠিক সময়ে পুনরায় অর্ডার দিতে ইনভেন্টরি ব্যবস্থাপনা জরুরি।'),
        ('গ্রস প্রফিট কী?','বিক্রয় মূল্য থেকে পণ্যের সরাসরি উৎপাদন খরচ বাদ দিলে গ্রস প্রফিট পাওয়া যায়।'),
        ('পার্টনারশিপ ব্যবসায় কী কী সুবিধা?','দায়িত্ব ভাগ, বেশি মূলধন, ভিন্ন দক্ষতার সমন্বয় এবং ঝুঁকি ভাগাভাগি সুবিধা।'),
        ('স্টার্টআপ ব্যর্থ হওয়ার কারণ কী?','বাজার চাহিদা না বোঝা, মূলধন সংকট, দলগত সমস্যা ও প্রতিযোগিতার মোকাবেলা না করা।'),
    ]
    _add_niche(biz_qa, 500)
    print(f'     ✓ ব্যবসা: {len(biz_qa)*500:,} লাইন')

    # ── ৫. প্রযুক্তি ও কম্পিউটার (১০,০০০) ──────────────────────────
    tech_qa = [
        ('Python কিভাবে শিখব?','W3Schools, Codecademy বা বাংলা ইউটিউব কোর্স দিয়ে শুরু করুন। প্রতিদিন ১ ঘণ্টা প্র্যাকটিস করুন।'),
        ('ওয়েব ডিজাইন ও ডেভেলপমেন্টের পার্থক্য?','ডিজাইন = চেহারা/UI। ডেভেলপমেন্ট = সেই ডিজাইনকে কোড দিয়ে কার্যকর করা।'),
        ('ফ্রিল্যান্সিং কিভাবে শুরু করব?','স্কিল শিখুন, পোর্টফোলিও তৈরি করুন, Fiverr/Upwork-এ প্রোফাইল করুন।'),
        ('সাইবার নিরাপত্তা কী?','কম্পিউটার, নেটওয়ার্ক ও ডেটাকে হ্যাকার থেকে রক্ষা করার পদ্ধতি।'),
        ('AI কী?','AI হলো মানুষের মতো চিন্তা করতে পারে এমন কম্পিউটার সিস্টেম। ChatGPT, Gemini উদাহরণ।'),
        ('ব্লকচেইন কী?','বিকেন্দ্রীভূত ডিজিটাল খাতা যেখানে তথ্য পরিবর্তন করা যায় না। Bitcoin এই প্রযুক্তিতে চলে।'),
        ('ক্লাউড কম্পিউটিং কী?','ইন্টারনেটের মাধ্যমে সার্ভার, স্টোরেজ ও সফটওয়্যার ব্যবহার করা। AWS, Google Cloud উদাহরণ।'),
        ('অ্যাপ ডেভেলপমেন্ট কোথায় শিখব?','Android: Kotlin/Java, iOS: Swift। YouTube, Udemy-তে বাংলা কোর্স পাবেন।'),
        ('ডেটা সায়েন্স কী?','বড় ডেটা বিশ্লেষণ করে তথ্য বের করার বিজ্ঞান। Python, R ও SQL প্রধান হাতিয়ার।'),
        ('VPN কী?','VPN ট্র্যাফিক এনক্রিপ্ট করে ও আইপি লুকায়, ফলে গোপনীয়তা রক্ষা পায়।'),
        ('GitHub কী?','GitHub হলো কোড সংরক্ষণ ও সহযোগিতার প্ল্যাটফর্ম যেখানে Git version control ব্যবহার হয়।'),
        ('API কী?','API (Application Programming Interface) হলো দুটি সফটওয়্যার পরস্পরের সাথে কথা বলার নিয়ম।'),
        ('Machine Learning কী?','ডেটা থেকে প্যাটার্ন শিখে ভবিষ্যৎ পূর্বানুমান করার কম্পিউটার বিজ্ঞান।'),
        ('SQL কী?','SQL (Structured Query Language) হলো ডেটাবেস থেকে তথ্য বের করা ও পরিচালনার ভাষা।'),
        ('স্মার্টফোন পরিষ্কার রাখার উপায়?','অপ্রয়োজনীয় অ্যাপ মুছুন, ক্যাশ ক্লিয়ার করুন, ভাইরাস স্ক্যান করুন, সফটওয়্যার আপডেট রাখুন।'),
        ('৫G প্রযুক্তি কী?','৫G হলো পঞ্চম প্রজন্মের মোবাইল নেটওয়ার্ক যা ৪G-এর চেয়ে ১০ গুণ দ্রুত।'),
        ('ইন্টারনেট অফ থিংস কী?','IoT হলো ঘরের ডিভাইস (ফ্রিজ, বাল্ব) ইন্টারনেটের মাধ্যমে নিয়ন্ত্রণ করার প্রযুক্তি।'),
        ('প্রোগ্রামিং ভাষা কোনটি প্রথমে শিখব?','সম্পূর্ণ নতুনদের জন্য Python সেরা। সহজ সিনট্যাক্স ও ব্যাপক ব্যবহার।'),
        ('ওয়েব হোস্টিং কী?','ওয়েবসাইটের ফাইল ইন্টারনেটে সংরক্ষণের সার্ভিস। Shared, VPS, Dedicated প্রকারভেদ আছে।'),
        ('ডিজিটাল সিগনেচার কী?','ডিজিটাল সিগনেচার ক্রিপ্টোগ্রাফিক পদ্ধতিতে নথির সত্যতা ও অখণ্ডতা নিশ্চিত করে।'),
    ]
    _add_niche(tech_qa, 500)
    print(f'     ✓ প্রযুক্তি: {len(tech_qa)*500:,} লাইন')

    # ── ৬. কৃষি ও গ্রামীণ জীবন (১০,০০০) ────────────────────────────
    agri_qa = [
        ('ধান চাষের সঠিক সময়?','বোরো: নভেম্বর-ডিসেম্বর বোনা, এপ্রিল কাটা। আমন: জুন-জুলাই বোনা, ডিসেম্বর কাটা।'),
        ('ফসলে সার কিভাবে ব্যবহার করব?','ইউরিয়া (নাইট্রোজেন), TSP (ফসফরাস), MOP (পটাশিয়াম) মাটি পরীক্ষা করে দিন।'),
        ('পোকামাকড় দমনের প্রাকৃতিক উপায়?','নিম তেল স্প্রে, সাথী ফসল, জৈব কীটনাশক ও আলোর ফাঁদ ব্যবহার করুন।'),
        ('মাছ চাষে কী খাওয়াব?','প্রাকৃতিক খাবার বাড়াতে জৈব সার দিন। সম্পূরক: ভাতের মাড়, খৈল, চাল কুঁড়া।'),
        ('গরু মোটাতাজাকরণে কী খাওয়াব?','UMS, ঘাস, ভুট্টা, চাল কুঁড়া, খৈল দিন। ৩-৪ মাসে গরু মোটা হবে।'),
        ('মাটি পরীক্ষা কেন দরকার?','মাটির pH ও পুষ্টি জানতে। সঠিক সারে ফলন ৩০-৪০% বাড়ে।'),
        ('সবজি চাষে সেচ কখন দেব?','সকাল/বিকেলে সেচ দিন। গ্রীষ্মে ৫-৭ দিন ও শীতে ১০-১৫ দিনে একবার।'),
        ('হাঁস-মুরগির রোগ প্রতিরোধ?','নিয়মিত টিকা (রানীক্ষেত, মারেক্স), খামার পরিষ্কার, পর্যাপ্ত আলো-বায়ু দিন।'),
        ('কম্পোস্ট সার কিভাবে তৈরি?','গোবর, ঘাস, উচ্ছিষ্ট স্তরে সাজান, আর্দ্র রাখুন, মাসে ২ বার নাড়ুন। ৩ মাসে তৈরি।'),
        ('কৃষি ঋণ কোথায় পাব?','বাংলাদেশ কৃষি ব্যাংক, রাজশাহী কৃষি উন্নয়ন ব্যাংক, ব্র্যাক বা গ্রামীণ ব্যাংক।'),
        ('হাইব্রিড বীজের সুবিধা কী?','হাইব্রিড বীজ বেশি ফলনশীল, রোগ প্রতিরোধী ও পরিবেশ সহনশীল।'),
        ('ছাদবাগান কিভাবে করব?','বড় পাত্র, ভালো মাটি ও সার দিয়ে টমেটো, মরিচ, লেটুস চাষ করুন। নিয়মিত পানি দিন।'),
        ('ভার্মি কম্পোস্ট কী?','কেঁচো ব্যবহার করে জৈব বর্জ্য থেকে তৈরি উন্নতমানের সার।'),
        ('মৌ চাষ কিভাবে করব?','মৌ বাক্স স্থাপন করুন, রানী মৌমাছি সংগ্রহ করুন। ফুলের মৌসুমে মধু সংগ্রহ করুন।'),
        ('সেচ পদ্ধতি কোনটি সবচেয়ে ভালো?','ড্রিপ সেচ সবচেয়ে সাশ্রয়ী ও কার্যকর। ৫০% পানি সাশ্রয় হয়।'),
        ('ফসলের রোগ চেনার উপায়?','পাতায় দাগ, হলুদ হওয়া, ঢলে পড়া ও ফল পচন দেখে রোগ চিহ্নিত করুন।'),
        ('চাষাবাদে লাভজনক ফসল কোনটি?','সবজি (টমেটো, বেগুন), ফুল, মশলা (আদা, রসুন) ও ফল চাষ লাভজনক।'),
        ('জলাবদ্ধতায় ফসল বাঁচানো?','দ্রুত পানি নিষ্কাশন করুন, সহনশীল জাত ব্যবহার করুন, উঁচু বেড তৈরি করুন।'),
        ('জৈব সবজি চাষ কিভাবে করব?','রাসায়নিক সার ও কীটনাশক বাদ দিন, কম্পোস্ট ও জৈব কীটনাশক ব্যবহার করুন।'),
        ('কৃষি বিমা কী?','প্রাকৃতিক দুর্যোগে ফসলের ক্ষতি হলে ক্ষতিপূরণ দেয় কৃষি বিমা। সরকার ভর্তুকি দেয়।'),
    ]
    _add_niche(agri_qa, 500)
    print(f'     ✓ কৃষি: {len(agri_qa)*500:,} লাইন')

    # ── ৭. ইসলামিক জ্ঞান (১০,০০০) ───────────────────────────────────
    islamic_qa = [
        ('নামাজ কয় ওয়াক্ত ও কত রাকাত?','পাঁচ ওয়াক্ত: ফজর ২, জোহর ৪, আসর ৪, মাগরিব ৩, এশা ৪ রাকাত ফরজ।'),
        ('রোজা কার উপর ফরজ?','প্রাপ্তবয়স্ক, সুস্থ মুসলমানের উপর রমজান মাসে রোজা ফরজ।'),
        ('যাকাতের নিসাব কত?','সাড়ে সাত তোলা সোনা বা সাড়ে বায়ান্ন তোলা রুপার সমমূল্যে ২.৫% যাকাত ফরজ।'),
        ('হজ কার উপর ফরজ?','সামর্থ্যবান মুসলমানের জীবনে একবার হজ ফরজ।'),
        ('তাহাজ্জুদ কখন পড়ব?','মধ্যরাতের পর থেকে ফজরের আজানের আগ পর্যন্ত। ন্যূনতম ২ রাকাত।'),
        ('ইসলামে পর্দার বিধান কী?','মাহরাম ছাড়া পুরুষের সামনে মুখ ও হাত ছাড়া আওরাত ঢেকে রাখা ওয়াজিব।'),
        ('সুদ কি হারাম?','হ্যাঁ, কোরআনে সুদ স্পষ্টভাবে হারাম।'),
        ('জানাজার নামাজ কিভাবে পড়ব?','চার তাকবিরে: ১ম ছানা, ২য় দরুদ, ৩য় দোয়া, ৪র্থ সালাম।'),
        ('হালাল ও হারামের পার্থক্য?','হালাল অনুমোদিত, হারাম নিষিদ্ধ। শুকরের মাংস ও মদ হারাম।'),
        ('কোরআন মুখস্থ করার পদ্ধতি?','প্রতিদিন নির্দিষ্ট আয়াত পড়ুন, অর্থ বুঝুন, অডিও শুনুন ও নামাজে পড়ুন।'),
        ('ইস্তেগফার কী?','আল্লাহর কাছে ক্ষমা চাওয়াকে ইস্তেগফার বলে। "আস্তাগফিরুল্লাহ" পড়া সুন্নত।'),
        ('দোয়া কবুলের শর্ত কী?','হালাল উপার্জন, একনিষ্ঠতা, ধৈর্য ও আল্লাহর উপর পূর্ণ আস্থা।'),
        ('ইসলামে সুদের বিকল্প কী?','মুরাবাহা, মুশারাকা, মুদারাবা ইসলামী ব্যাংকিং পদ্ধতিতে সুদমুক্ত বিকল্প।'),
        ('কুরবানি কার উপর ওয়াজিব?','নিসাব পরিমাণ সম্পদের মালিক প্রতিটি মুসলমানের উপর কুরবানি ওয়াজিব।'),
        ('ইতিকাফ কী?','রমজানের শেষ দশ দিন মসজিদে ইবাদতের উদ্দেশ্যে অবস্থান করাকে ইতিকাফ বলে।'),
        ('নফল নামাজ কী কী?','তাহাজ্জুদ, ইশরাক, চাশত, আওয়াবিন, তাহিয়্যাতুল মসজিদ প্রধান নফল নামাজ।'),
        ('ইসলামে বিয়ের নিয়ম কী?','ওলি, সাক্ষী, মোহর ও উভয়ের সম্মতি বিয়ের মূল শর্ত। কাজী সম্পাদন করেন।'),
        ('মিরাজ কী?','নবী মুহাম্মদ (সা.) মক্কা থেকে বায়তুল মুকাদ্দাস ও সাত আসমান ভ্রমণের মিরাজ।'),
        ('ইসলামে দান-খয়রাতের গুরুত্ব?','দান পাপ মোচন করে, সম্পদ বরকতময় হয় এবং সমাজে ভারসাম্য আসে।'),
        ('তওবা কিভাবে করব?','পাপ ছেড়ে দিন, অনুতাপ করুন, আর না করার সংকল্প করুন এবং আল্লাহর কাছে মাফ চান।'),
    ]
    _add_niche(islamic_qa, 500)
    print(f'     ✓ ইসলামিক: {len(islamic_qa)*500:,} লাইন')

    # ── ৮. রান্না ও রেসিপি (১০,০০০) ─────────────────────────────────
    recipe_qa = [
        ('মুরগির ঝোল রান্নার পদ্ধতি?','মুরগি মেরিনেট করুন, তেলে পেঁয়াজ ভাজুন, মশলা কষুন, মুরগি দিন, পানি দিয়ে সিদ্ধ করুন।'),
        ('বিরিয়ানিতে কোন মশলা লাগে?','এলাচ, দারচিনি, লবঙ্গ, জায়ফল, জয়িত্রী, তেজপাতা, স্টার অ্যানিস, কেওড়া ও জাফরান।'),
        ('ভাত নরম করার উপায়?','চাল ধুয়ে ৩০ মিনিট ভিজিয়ে, সঠিক পানি (১:২) দিয়ে রান্না করুন।'),
        ('হালুয়া কিভাবে তৈরি?','সুজি ঘিয়ে ভাজুন, চিনির শরবত দিন, দমে রাখুন, এলাচ-বাদাম দিন।'),
        ('ভাপা পিঠা কিভাবে বানাব?','চালের গুঁড়া, নারিকেল, গুড় মিশিয়ে ছাঁচে ভরুন, ভাপে সিদ্ধ করুন।'),
        ('শীতের সবজির তরকারি?','গাজর, ফুলকপি, মটরশুঁটি কেটে, তেলে পেঁয়াজ-মশলা ভেজে সবজি দিন, লবণ ও পানি দিন।'),
        ('দই কিভাবে তৈরি?','দুধ ফুটিয়ে ৪০-৪৫°C-এ ঠাণ্ডা করুন, টক দই মিশিয়ে ৬-৮ ঘণ্টা গরম জায়গায় রাখুন।'),
        ('মাছের কোফতা কিভাবে বানাব?','মাছ সিদ্ধ, কাঁটা বেছে নিন, মশলা মিশিয়ে গোল্লা করুন, ভেজে গ্রেভিতে দিন।'),
        ('লাচ্ছি বানানোর পদ্ধতি?','দই, ঠাণ্ডা পানি, চিনি বা লবণ ও এলাচ ব্লেন্ড করুন, বরফ দিয়ে পরিবেশন করুন।'),
        ('ডাল রান্নায় কী মশলা ব্যবহার করব?','হলুদ, মরিচ, আদা, রসুন দিয়ে সিদ্ধ করুন। শেষে মরিচ-পেঁয়াজ-রসুন ভাজা দিন।'),
        ('চিকেন রোস্ট কিভাবে রান্না করব?','মুরগি মেরিনেট করুন (দই, মশলা), ওভেনে ১৮০°C-এ ৪৫ মিনিট বা কড়াইয়ে রান্না করুন।'),
        ('সেমাই রান্নার পদ্ধতি?','দুধ ফুটিয়ে সেমাই দিন, চিনি-এলাচ দিন, মাখন দিয়ে নাড়ুন, ঘন হলে নামান।'),
        ('শাহি টুকরা কিভাবে তৈরি?','পাউরুটি ঘিয়ে ভাজুন, দুধের রাবড়ি তৈরি করুন, রুটি ডুবিয়ে পিস্তা-বাদাম দিয়ে পরিবেশন।'),
        ('আলু ভর্তা কিভাবে বানাব?','আলু সিদ্ধ করুন, চটকে নিন, কাঁচা মরিচ, পেঁয়াজ, সরিষার তেল ও লবণ মিশান।'),
        ('পাঁচ মিশালি তরকারি কিভাবে?','পটল, বেগুন, করলা, ঝিঙে, পেঁপে কেটে আলাদা ভেজে একসাথে মশলায় কষান।'),
        ('রসগোল্লা কিভাবে তৈরি?','ছানা ভালো চটকে নিন, গোল্লা করুন, চিনির সিরায় সিদ্ধ করুন।'),
        ('শিঙাড়া কিভাবে বানাব?','ময়দার খামির, আলু-মটর পুর ভরুন, তেলে ভাজুন।'),
        ('সবজি স্যুপ কিভাবে?','গাজর, বাঁধাকপি, মটরশুঁটি সিদ্ধ করুন, লবণ-গোলমরিচ-আদা দিন, ব্লেন্ড করুন।'),
        ('মোগলাই পরোটা কিভাবে?','ময়দার পাতলা রুটিতে ডিম-পেঁয়াজ-মশলা ঢেলে ভাঁজ করে তেলে ভাজুন।'),
        ('খিচুড়ি কিভাবে রান্না করব?','চাল-ডাল ধুয়ে হলুদ-মশলা দিয়ে রান্না করুন, সবজি ও ঘি দিন, দমে রাখুন।'),
    ]
    _add_niche(recipe_qa, 500)
    print(f'     ✓ রান্না: {len(recipe_qa)*500:,} লাইন')

    # ── ৯. আইন ও অধিকার (১০,০০০) ───────────────────────────────────
    law_qa = [
        ('জমি রেজিস্ট্রেশন কিভাবে?','সাব-রেজিস্ট্রার অফিসে দলিল, NID, খতিয়ান নিয়ে স্ট্যাম্প ও ফি দিয়ে রেজিস্ট্রেশন করুন।'),
        ('তালাক দেওয়ার নিয়ম?','তালাকনামা লিখে চেয়ারম্যান/মেয়র অফিসে জমা দিন। ৯০ দিন পর কার্যকর।'),
        ('ভোক্তা অধিকার লঙ্ঘনে কোথায় যাব?','DNCRP-এ অভিযোগ করুন। হটলাইন: ১৬১২১।'),
        ('শ্রমিকের ন্যূনতম মজুরি?','পোশাক শিল্পে ন্যূনতম ১২,৫০০ টাকা। অন্যত্র মজুরি বোর্ড নির্ধারণ করে।'),
        ('সাইবার অপরাধে কী করব?','থানায় মামলা বা Cyber Tribunal-এ যান। পুলিশের সাইবার ক্রাইম ইউনিটে অভিযোগ দিন।'),
        ('নারী নির্যাতনে কোথায় অভিযোগ?','থানায় GD বা মামলা করুন। হেল্পলাইন: ১০৯।'),
        ('জমির খতিয়ান কী?','মালিকের নাম, দাগ নম্বর, পরিমাণ ও ধরন সম্বলিত সরকারি নথি।'),
        ('উত্তরাধিকার সম্পত্তি কিভাবে ভাগ?','মুসলিম আইনে পুত্র মেয়ের দ্বিগুণ পায়। সন্তান না থাকলে স্বামী স্ত্রীর অর্ধেক।'),
        ('পাসপোর্ট কিভাবে করব?','www.dip.gov.bd-এ অনলাইনে আবেদন, ফি দিন, পাসপোর্ট অফিসে যান।'),
        ('ব্যাংক ঋণে কী লাগে?','NID, ছবি, আয়ের প্রমাণ, ব্যাংক স্টেটমেন্ট ও জামানত (জমি বা গ্যারেন্টর) লাগে।'),
        ('জমি কেনার আগে কী যাচাই করব?','মালিকানা দলিল, খতিয়ান, দাগ নম্বর, মৌজা ম্যাপ ও ঋণমুক্ত কিনা যাচাই করুন।'),
        ('বিবাহ রেজিস্ট্রেশন কেন দরকার?','আইনি সুরক্ষা, সম্পত্তির অধিকার ও বিদেশে ভিসার জন্য বিবাহ রেজিস্ট্রেশন জরুরি।'),
        ('চুরি হলে কী করব?','থানায় GD করুন, প্রমাণ সংগ্রহ করুন ও বীমা থাকলে ক্লেইম করুন।'),
        ('কর্মক্ষেত্রে হয়রানি হলে?','HR-এ লিখিত অভিযোগ করুন। না হলে শ্রম আদালতে মামলা করুন।'),
        ('ভাড়াটিয়ার অধিকার কী?','বৈধ চুক্তি ছাড়া উচ্ছেদ করা যাবে না। অযৌক্তিক ভাড়া বৃদ্ধি প্রতিরোধের আইন আছে।'),
        ('মামলার খরচ কিভাবে বহন করব?','সরকারি আইনি সহায়তা (NLASO) বিনামূল্যে সেবা দেয়। আবেদন করুন।'),
        ('NID হারালে কী করব?','নিকটস্থ উপজেলা নির্বাচন অফিসে গিয়ে পুনরায় আবেদন করুন।'),
        ('জন্ম নিবন্ধন কিভাবে করব?','ইউনিয়ন পরিষদ বা পৌরসভায় আবেদন করুন। হাসপাতালের নথি ও বাবা-মার NID লাগে।'),
        ('মৃত্যু সনদ কীভাবে পাব?','স্থানীয় ইউনিয়ন পরিষদ বা পৌরসভায় আবেদন করুন। মৃত্যুর কারণ ও তারিখ দিন।'),
        ('আদালতের রায়ের বিরুদ্ধে আপিল?','নিম্ন আদালতের রায়ের ৩০ দিনের মধ্যে উচ্চ আদালতে আপিল করতে পারবেন।'),
    ]
    _add_niche(law_qa, 500)
    print(f'     ✓ আইন: {len(law_qa)*500:,} লাইন')

    # ── ১০. সাধারণ জ্ঞান ও বাংলাদেশ (১০,০০০) ────────────────────────
    gk_qa = [
        ('বাংলাদেশের রাজধানী কোথায়?','বাংলাদেশের রাজধানী ঢাকা, বুড়িগঙ্গা নদীর তীরে অবস্থিত।'),
        ('বাংলাদেশের জাতীয় সংগীত কী?','আমার সোনার বাংলা — রবীন্দ্রনাথ ঠাকুরের লেখা, প্রথম ১০ লাইন জাতীয় সংগীত।'),
        ('পদ্মা সেতু কত কিলোমিটার?','পদ্মা সেতুর দৈর্ঘ্য ৬.১৫ কিলোমিটার, ২০২২ সালে উদ্বোধন।'),
        ('বাংলাদেশের জনসংখ্যা কত?','প্রায় ১৭ কোটি (২০২৩)। বিশ্বের জনঘনত্বে অন্যতম।'),
        ('সুন্দরবন কোথায়?','খুলনা, সাতক্ষীরা ও বাগেরহাটে। বিশ্বের বৃহত্তম ম্যানগ্রোভ বন।'),
        ('বাংলাদেশের প্রধান নদী কোনটি?','পদ্মা, যমুনা ও মেঘনা। এদের মিলনস্থলে বৃহত্তম মোহনা।'),
        ('ঢাকা বিশ্ববিদ্যালয় কবে প্রতিষ্ঠিত?','১৯২১ সালের ১ জুলাই। বাংলাদেশের সবচেয়ে প্রাচীন বিশ্ববিদ্যালয়।'),
        ('বাংলাদেশের মুদ্রার নাম?','টাকা (BDT)। ১ টাকা = ১০০ পয়সা।'),
        ('কক্সবাজার কোন জেলায়?','চট্টগ্রাম বিভাগে। বিশ্বের দীর্ঘতম সমুদ্র সৈকত (১২০ কিমি)।'),
        ('বাংলাদেশের প্রথম রাষ্ট্রপতি কে?','বঙ্গবন্ধু শেখ মুজিবুর রহমান।'),
        ('বাংলাদেশে প্রথম শিল্পকারখানা?','নারায়ণগঞ্জে আদমজী জুট মিল, ১৯৫১ সালে স্থাপিত।'),
        ('বাংলাদেশের জাতীয় খেলা কী?','কাবাডি বাংলাদেশের জাতীয় খেলা।'),
        ('রবীন্দ্রনাথ ঠাকুর কবে জন্মগ্রহণ করেন?','১৮৬১ সালের ৭ মে (২৫ বৈশাখ) কলকাতায়।'),
        ('বাংলাদেশের সর্বোচ্চ পর্বত কোনটি?','তাজিংডং (বিজয়) ১,২৩১ মিটার উচ্চ, বান্দরবানে অবস্থিত।'),
        ('বাংলাদেশে কয়টি বিভাগ আছে?','৮টি বিভাগ: ঢাকা, চট্টগ্রাম, রাজশাহী, খুলনা, বরিশাল, সিলেট, রংপুর, ময়মনসিংহ।'),
        ('বায়তুল মোকাররম কী?','ঢাকায় অবস্থিত বাংলাদেশের জাতীয় মসজিদ।'),
        ('ছয় দফা কী?','১৯৬৬ সালে বঙ্গবন্ধু ঘোষিত বাঙালির স্বায়ত্তশাসনের দাবিনামা।'),
        ('বাংলাদেশের কোন নদী সবচেয়ে দীর্ঘ?','মেঘনা নদী বাংলাদেশে সবচেয়ে দীর্ঘ।'),
        ('জাতিসংঘে বাংলাদেশ কবে যোগ দেয়?','১৯৭৪ সালের ১৭ সেপ্টেম্বর।'),
        ('বাংলাদেশের আইনসভার নাম কী?','জাতীয় সংসদ। আসন সংখ্যা ৩৫০ (৩০০ সরাসরি + ৫০ সংরক্ষিত মহিলা)।'),
    ]
    _add_niche(gk_qa, 500)
    print(f'     ✓ সাধারণ জ্ঞান: {len(gk_qa)*500:,} লাইন')

    total_niche = (len(dm_qa)+len(nctb_qa)+len(health_qa)+len(biz_qa)+len(tech_qa)+
                   len(agri_qa)+len(islamic_qa)+len(recipe_qa)+len(law_qa)+len(gk_qa)) * 500
    print(f'  ✅ মোট নিশ ডোমেন ডেটা: {total_niche:,} লাইন')

    # ━━━ SOURCE ৬: পাটিগণিত (১,০০,০০০) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print('  [৬/৬] পাটিগণিত + জেনারেল গণিত → ১,০০,০০০...')
    random.seed(42)
    math_types = ['add','sub','mul','div','percent','unitary','profit','en_math']
    for _ in range(100000):
        m = random.choice(math_types)
        if m == 'add':
            a,b = random.randint(5,9999),random.randint(5,9999)
            q,ans = f'{a} এর সাথে {b} যোগ করলে কত হয়?', f'{a} + {b} = {a+b}।'
        elif m == 'sub':
            a,b = random.randint(50,9999),random.randint(5,4999)
            if a<b: a,b=b,a
            q,ans = f'{a} থেকে {b} বিয়োগ করলে কত থাকে?', f'{a} - {b} = {a-b}।'
        elif m == 'mul':
            a,b = random.randint(2,999),random.randint(2,99)
            q,ans = f'{a} কে {b} দিয়ে গুণ করলে কত?', f'{a} × {b} = {a*b}।'
        elif m == 'div':
            d = random.randint(2,30); qv = d*random.randint(2,100)
            q,ans = f'{qv} কে {d} দিয়ে ভাগ করলে কত?', f'{qv} ÷ {d} = {qv//d}।'
        elif m == 'percent':
            base = random.choice([50,100,200,500,1000,2000,5000])
            rate = random.choice([5,10,15,20,25,30,50])
            val  = int(base*rate/100)
            q,ans = f'{base} টাকার {rate}% কত?', f'{base} × {rate}/100 = {val} টাকা।'
        elif m == 'unitary':
            n1,up = random.randint(2,8),random.randint(5,50)
            c1,n2 = n1*up,random.randint(9,20); c2=n2*up
            q,ans = (f'{n1}টি জিনিসের দাম {c1} টাকা হলে {n2}টির দাম কত?',
                     f'১টির দাম {c1}÷{n1}={up} টাকা। {n2}টির দাম {up}×{n2}={c2} টাকা।')
        elif m == 'profit':
            cp = random.randint(100,5000); prof=random.randint(10,cp//2); sp=cp+prof
            q,ans = (f'একটি জিনিস {cp} টাকায় কিনে {sp} টাকায় বিক্রি করলে লাভ কত?',
                     f'লাভ = {sp} - {cp} = {prof} টাকা।')
        else:
            a,b = random.randint(2,99),random.randint(2,50)
            q,ans = f'What is {a} multiplied by {b}?', f'{a} × {b} = {a*b}.'
        corpus_lines.append(f'{q} {ans}')
        sft_lines.append(f'প্রশ্ন: {q} উত্তর: {ans} <EOS>')
    print(f'     ✓ গণিত: 1,00,000 লাইন')

    # ━━━ শাফেল ও ফাইলে লেখা ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    print('⚡ শাফেল ও ফাইল লেখা হচ্ছে...')
    random.shuffle(corpus_lines)
    random.shuffle(sft_lines)
    with open(corpus_path,'w',encoding='utf-8') as f:
        for l in corpus_lines: f.write(l+'\n')
    with open(sft_data_path,'w',encoding='utf-8') as f:
        for l in sft_lines: f.write(l+'\n')
    for old in ['data/corpus_tokens.bin','data/sft_data_tokens.bin']:
        if os.path.exists(old): os.remove(old)
    c_mb = os.path.getsize(corpus_path)/(1024*1024)
    s_mb = os.path.getsize(sft_data_path)/(1024*1024)
    print('='*70)
    print(f'✅ CORPUS  : {len(corpus_lines):>10,} লাইন  ({c_mb:.1f} MB)')
    print(f'✅ SFT     : {len(sft_lines):>10,} লাইন  ({s_mb:.1f} MB)')
    print('='*70)
    # ━━━ লাইভ স্টেপ ক্যালকুলেশন ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    BATCH=4; GRAD=4; BLOCK=512; AVG_TOK=30; SPLIT=0.9; EP=2
    est_tok = len(corpus_lines)*AVG_TOK*SPLIT
    tps = BATCH*GRAD*BLOCK
    spe = int(est_tok/tps)
    total = spe*EP
    print(f'\n📊 স্টেজ-১ স্টেপ ক্যালকুলেশন:')
    print(f'   মোট লাইন         : {len(corpus_lines):,}')
    print(f'   আনুমানিক টোকেন  : {int(est_tok):,}')
    print(f'   Tokens/Step      : {tps:,}')
    print(f'   Steps/Epoch      : {spe:,}')
    print(f'   মোট স্টেপ (২ ep) : {total:,}')
    print(f'   সময় (T4 ~3.5it/s): ~{total//3//60} মিনিট')


In [ ]:
# Step 5: টোকেনাইজার তৈরি
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
special_tokens = ['<PAD>','<UNK>','<BOS>','<EOS>','<|system|>','<|user|>','<|assistant|>','<|math|>']
tok = Tokenizer(models.BPE(unk_token='<UNK>'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=10000, special_tokens=special_tokens,
                               min_frequency=2, show_progress=True)
print('⚡ ১১ লাখ লাইনে টোকেনাইজার ট্রেনিং...')
tok.train(['data/corpus.txt','data/sft_data.txt'], trainer)
tok.save('tokenizer.json')
print(f'✓ Vocab: {tok.get_vocab_size():,}')

In [ ]:
# Step 6: 🔥 Stage 1 — Pretraining (2 epochs ~7,250 steps)
import os, glob, time, math, torch
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

tokenizer      = Tokenizer.from_file('tokenizer.json')
dataset_stage1 = BengaliDataset(corpus_path='data/corpus.txt', tokenizer=tokenizer,
                                 block_size=GPTConfig.block_size, split_ratio=0.9)
tps  = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
spe  = dataset_stage1.train_len // tps
EPOCHS = 2.0
max_iters = int(spe * EPOCHS)

print(f'📊 Stage-1:')
print(f'   Train Tokens  : {dataset_stage1.train_len:,}')
print(f'   Tokens/Step   : {tps:,}')
print(f'   Steps/Epoch   : {spe:,}')
print(f'   Total (2 ep)  : {max_iters:,}')
print(f'   ~Time (T4)    : ~{max_iters//3//60} min')

raw_model = GPT(GPTConfig).to(device)
try: model = torch.compile(raw_model); print('✓ compile')
except: model = raw_model

optimizer = torch.optim.AdamW(raw_model.parameters(),
    lr=GPTConfig.learning_rate, betas=(0.9,0.95), weight_decay=0.1)
scaler = GradScaler()

start = 1
ckpts = glob.glob(os.path.join(STAGE1_DIR,'stage1_step_*.pt'))
if ckpts:
    def _s(p):
        try: return int(p.split('_step_')[-1].replace('.pt',''))
        except: return 0
    lc = sorted(ckpts, key=_s)[-1]; ls = _s(lc)
    if 0 < ls < max_iters:
        raw_model.load_state_dict(torch.load(lc, map_location=device))
        start = ls+1; print(f'🔄 Resume step {start:,}')

def get_lr(it, mx, lr=3e-4, mlr=3e-5):
    w=400
    if it<w: return lr*it/w
    if it>mx: return mlr
    return mlr+.5*(1+math.cos(math.pi*(it-w)/(mx-w)))*(lr-mlr)

print(f'{'='*65}'); print(f'🔥 Stage-1: {start:,} → {max_iters:,}')
model.train(); optimizer.zero_grad(set_to_none=True); t0=time.time()

for step in range(start, max_iters+1):
    lr = get_lr(step, max_iters, GPTConfig.learning_rate, GPTConfig.min_lr)
    for g in optimizer.param_groups: g['lr']=lr
    acc=0.0
    for _ in range(GPTConfig.gradient_accumulation_steps):
        x,y = dataset_stage1.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            _,loss = model(x, targets=y); loss = loss/GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward(); acc+=loss.item()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    if step%250==0 or step==start:
        ep=(step*tps)/dataset_stage1.train_len; ela=time.time()-t0
        spd=(step-start+1)/ela if ela>0 else 0; eta=(max_iters-step)/spd/60 if spd>0 else 0
        print(f'[S1] {step:5d}/{max_iters} Ep{ep:.2f} | Loss:{acc:.4f} | LR:{lr:.2e} | {spd:.2f}it/s ETA:{eta:.1f}m')
    if step%500==0 or step==max_iters:
        torch.save(raw_model.state_dict(), os.path.join(STAGE1_DIR,f'stage1_step_{step}.pt'))

final1=os.path.join(DRIVE_BASE_DIR,'checkpoint_stage_1.pt')
torch.save(raw_model.state_dict(), final1)
print(f'🎉 Stage 1 সম্পন্ন! → {final1}')

In [ ]:
# Step 7: 🎯 Stage 2 — SFT (3 epochs)
import os, glob, time, math, torch
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
s1 = os.path.join(DRIVE_BASE_DIR,'checkpoint_stage_1.pt')
raw_sft = GPT(GPTConfig).to(device)
if os.path.exists(s1):
    raw_sft.load_state_dict(torch.load(s1, map_location=device))
    print(f'✓ Stage-1 লোড: {s1}')
else: print('⚠️ Stage-1 নেই, scratch থেকে!')

try: sft_model=torch.compile(raw_sft); print('✓ compile')
except: sft_model=raw_sft

tokenizer      = Tokenizer.from_file('tokenizer.json')
dataset_stage2 = BengaliDataset(corpus_path='data/sft_data.txt', tokenizer=tokenizer,
                                 block_size=GPTConfig.block_size, split_ratio=0.9)
tps  = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
spe  = max(1, dataset_stage2.train_len // tps)
EPOCHS2=3.0; max_iters2=int(spe*EPOCHS2)

print(f'📊 Stage-2: Tokens={dataset_stage2.train_len:,} | Steps/Ep={spe:,} | Total={max_iters2:,}')

optimizer=torch.optim.AdamW(raw_sft.parameters(),
    lr=GPTConfig.sft_learning_rate, betas=(0.9,0.95), weight_decay=0.1)
scaler=GradScaler()

start2=1
sft_ckpts=glob.glob(os.path.join(STAGE2_DIR,'stage2_step_*.pt'))
if sft_ckpts:
    def _ss(p):
        try: return int(p.split('_step_')[-1].replace('.pt',''))
        except: return 0
    lc2=sorted(sft_ckpts,key=_ss)[-1]; ls2=_ss(lc2)
    if 0<ls2<max_iters2:
        raw_sft.load_state_dict(torch.load(lc2,map_location=device))
        start2=ls2+1; print(f'🔄 SFT Resume {start2:,}')

def get_sft_lr(it,mx,lr=1e-4,mlr=1e-5):
    w=min(100,mx//10)
    if it<w: return lr*it/w
    if it>mx: return mlr
    return mlr+.5*(1+math.cos(math.pi*(it-w)/max(1,mx-w)))*(lr-mlr)

print(f'🔥 Stage-2: {start2:,} → {max_iters2:,}')
sft_model.train(); optimizer.zero_grad(set_to_none=True); t0=time.time()

for step in range(start2, max_iters2+1):
    lr=get_sft_lr(step,max_iters2,GPTConfig.sft_learning_rate,GPTConfig.sft_min_lr)
    for g in optimizer.param_groups: g['lr']=lr
    acc=0.0
    for _ in range(GPTConfig.gradient_accumulation_steps):
        x,y=dataset_stage2.get_batch('train',batch_size=GPTConfig.batch_size,device=device)
        with autocast(dtype=torch.float16):
            _,loss=sft_model(x,targets=y); loss=loss/GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward(); acc+=loss.item()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_sft.parameters(), 1.0)
    scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
    if step%25==0 or step==start2 or step==max_iters2:
        ep=(step*tps)/dataset_stage2.train_len; ela=time.time()-t0
        spd=(step-start2+1)/ela if ela>0 else 0
        print(f'[S2] {step:4d}/{max_iters2} Ep{ep:.2f} | Loss:{acc:.4f} | LR:{lr:.2e} | {spd:.2f}it/s')
    if step%100==0 or step==max_iters2:
        torch.save(raw_sft.state_dict(), os.path.join(STAGE2_DIR,f'stage2_step_{step}.pt'))

final2=os.path.join(DRIVE_BASE_DIR,'checkpoint_stage_2_final.pt')
torch.save(raw_sft.state_dict(), final2)
print(f'🎉 চূড়ান্ত মডেল → {final2}')

In [ ]:
# Step 8: 💬 চ্যাটবট টেস্ট
import torch
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
final2 = os.path.join(DRIVE_BASE_DIR,'checkpoint_stage_2_final.pt')
chat_model = GPT(GPTConfig).to(device)
chat_model.load_state_dict(torch.load(final2, map_location=device))
chat_model.eval()
tokenizer = Tokenizer.from_file('tokenizer.json')

# ✏️ প্রশ্ন লিখুন:
user_question = 'ডিজিটাল মার্কেটিং কী?'

prompt = f'প্রশ্ন: {user_question} উত্তর:'
enc    = tokenizer.encode(prompt)
ids    = enc.ids if hasattr(enc,'ids') else enc
inp    = torch.tensor([ids], dtype=torch.long, device=device)
eos_id = tokenizer.token_to_id('<EOS>')

with torch.no_grad():
    out = chat_model.generate(inp, max_new_tokens=200,
                               temperature=0.7, top_k=40,
                               repetition_penalty=1.25, eos_id=eos_id)
reply = tokenizer.decode(out[0].cpu().tolist()).split('<EOS>')[0].strip()
print('='*60)
print(reply)
print('='*60)